# Orsini Bayesian Langmuir Probe Demo

This notebook walks one local sample trace through the new Bayesian Orsini-style fit, the legacy heuristic pipeline, and the comparison summary.

In [ ]:
from pathlib import Path

import pandas as pd

from orsini_lp import compare_trace, fit_bayesian, load_manifest, load_trace, run_legacy
from orsini_lp.inference import BayesianConfig
from orsini_lp.plotting import plot_eedf_fit, plot_iv_fit

project_root = Path.cwd()
if not (project_root / 'sample_data' / 'local_manifest.csv').exists():
    project_root = project_root.parent

manifest_path = project_root / 'sample_data' / 'local_manifest.csv'
manifest = load_manifest(manifest_path)
manifest['manifest_path'] = manifest.attrs['manifest_path']
manifest[['trace_id', 'trace_path', 'file_format', 'flow_sccm', 'discharge_current_a']]

In [ ]:
trace_id = 'kr_10sccm_10a'
trace_row = manifest.loc[manifest['trace_id'] == trace_id].iloc[0]
trace = load_trace(trace_row)

config = BayesianConfig(nlive=80, dlogz=1.0, max_points=100, posterior_draws=120, random_seed=21)
bayes = fit_bayesian(trace, config=config)
legacy = run_legacy(trace)
comparison = compare_trace(trace, bayes, legacy)

comparison.comparison_table

In [ ]:
bayes.summary.loc[['Vp', 'Te', 'ne', 'p', 'n_qn']]

In [ ]:
iv_fig = plot_iv_fit(trace, bayes, legacy)
eedf_fig = plot_eedf_fit(bayes)
iv_fig, eedf_fig